# Delta Lake Training Notebook

Hands-on SQL guide to Delta Lake using the **TPC-H benchmark dataset** (7.5 M orders, 750 K customers) from the shared `samples.tpch` catalog. **All exercises are pure SQL; no PySpark required.**

| # | Topic | What you will learn |
|---|---|---|
| 1 | Create | CTAS, DDL, INSERT |
| 2 | DML | UPDATE, DELETE, MERGE |
| 3 | Storage, Delta Log & Checkpoints | File layout, commit JSON files, checkpoints, DESCRIBE HISTORY |
| 4 | Time Travel | VERSION AS OF, TIMESTAMP AS OF, @v syntax, RESTORE |
| 5 | Change Data Feed | delta.enableChangeDataFeed, table_changes() |
| 6 | Constraints | NOT NULL, CHECK, PRIMARY / FOREIGN KEY |
| 7 | Indexes & Data Skipping | File statistics, Bloom filter indexes, ANALYZE |
| 8 | Performance Techniques | Partitioning, Z-Order, Liquid Clustering, compaction, optimized writes, predictive optimization, key table properties |
| 9 | VACUUM | Retention, DRY RUN, LITE vs FULL |

**Working tables** (created in `one_dbx_dev.default`):
- `tpc_orders` — main table for DML, versions, and time travel
- `tpc_orders_partitioned` — partitioned by order year
- `tpc_orders_clustered` — managed by Liquid Clustering
- `tpc_log_demo` — tiny table to illustrate commits and checkpoints
- `tpc_orders_cdf` — Change Data Feed, constraints and indexes
- `tpc_customer` — parent table for primary / foreign keys
- `tpc_orders_small` — small-file problem, compaction and optimized writes

##DataLake
* Need for an organization to collect data from various sources of structured/ semistructured and unstructured data types
* Usage of these infromations in Bigdata and datascience applications
* Scalabilty and cost effective
<img src="/Volumes/training/bronze/images/Screenshot 2026-10-17 at 16.22.04.png" alt="Data Lake Sources" width="800"/>

##DELTALAKE = Datalake + ACID (Datalog and Schema Enforcement ​)

* A — ATOMICITY​
DELTA LAKE WRITES ALL FILE CHANGES AS A SINGLE ATOMIC COMMIT IN THE _DELTA_LOG, SO EITHER THE ENTIRE TRANSACTION SUCCEEDS OR NOTHING IS APPLIED.​
* C — CONSISTENCY​
SCHEMA ENFORCEMENT AND CONSTRAINTS ENSURE EVERY COMMIT RECORDED IN THE _DELTA_LOG KEEPS THE TABLE IN A VALID, CONSISTENT STATE.​
* I — ISOLATION​
 USING CONCURRENCY CONTROL EACH WRITER/READER SEES A CONSISTENT VERSION OF THE TABLE WITHOUT INTERFERING WITH OTHERS.​
* D — DURABILITY​
COMMITTED CHANGES ARE PERMANENTLY STORED IN THE _DELTA_LOG AND UNDERLYING STORAGE, GUARANTEEING RECOVERY EVEN AFTER FAILURES.​


## Delta Lake Architecture

Delta Lake sits as an **open-source storage layer** on top of your existing data lake (cloud object storage such as ADLS, S3, or GCS). It brings reliability, performance, and governance to data lakes.

### Key Architectural Components

| Component | Description |
|---|---|
| **Parquet Data Files** | The actual data is stored as Parquet files in cloud object storage, providing efficient columnar compression and encoding. |
| **Delta Transaction Log (`_delta_log/`)** | A sequentially ordered record of every transaction (commit) ever performed on the table. This is the single source of truth for the table's state. |
| **Checkpoint Files** | Periodic Parquet-format snapshots of the entire log state (every 10 commits by default), enabling fast log replay without reading all individual JSON commits. |
| **Deletion Vectors** | Lightweight `.bin` sidecar files that mark individual rows as deleted, avoiding expensive full-file rewrites for UPDATE/DELETE operations. |




### Write Path
1. **Write new Parquet data files** to cloud storage.
2. **Attempt to commit** a new JSON file to `_delta_log/` containing `add` and `remove` actions.
3. **Optimistic concurrency control** — if another writer committed first, Delta checks for conflicts; if none, it retries the commit with the next version number.
4. **Checkpoint** — every 10 commits, a checkpoint file is written to accelerate future reads.

### Delta Lake vs. Plain Data Lake

| Capability | Plain Data Lake | Delta Lake |
|---|---|---|
| ACID Transactions | ✗ | ✓ |
| Schema Enforcement | ✗ | ✓ |
| Time Travel | ✗ | ✓ (via versioned commits) |
| Scalable Metadata | ✗ | ✓ (transaction log + checkpoints) |
| Unified Batch & Streaming | ✗ | ✓ |
| Data Skipping | ✗ | ✓ (file-level statistics) |
| UPDATE / DELETE / MERGE | ✗ | ✓ |

##Columnar Storage 
* Data inside parquet file stored in columnar format unlike row based traditional database storage
* Provides better compression
* Avoid reading entire columns when only few columns are needed
<img src="/Volumes/training/bronze/images/Screenshot 2026-10-17 at 16.40.52.png" alt="Data Lake Sources" width="800"/>

## Setup 
Create schema bronze_<yourname> 
syntax:
CREATE SCHEMA IF NOT EXISTS training.bronze_<yourname>
MANAGED LOCATION 'abfss://training@unityms.dfs.core.windows.net/bronze<yourname>';
Set the working catalog and schema. 

In [0]:
%sql
USE CATALOG training;

In [0]:

%sql USE SCHEMA bronze;
 SELECT current_catalog() AS catalog, current_schema() AS schema;

catalog,schema
training,bronze


## 1. Creating Delta Tables and Inserting Records
**CTAS** (`CREATE TABLE ... AS SELECT`) copies data from the shared catalog into our own managed Delta table. We load 1992–1997 first, then `INSERT` 1998 to create a second version.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_orders
COMMENT 'Training copy of samples.tpch.orders'
AS
SELECT * FROM samples.tpch.orders
WHERE year(o_orderdate) <= 1997;

num_affected_rows,num_inserted_rows


## Delta Log

* Deltalog or the trasaction log is heart of the system
* It contains the information about files created or removed during the write
* Contains min,max,null values for first 32 columns of each data files 
* Once commit is successful a new deltalog file will be created with new version number 

Check the storage location
https://portal.azure.com/#@one-consultants.com/resource/subscriptions/11ebd399-8f6c-46af-b26f-71fc4572151b/resourceGroups/ONE-Azure/providers/Microsoft.Storage/storageAccounts/unityms/storagebrowser 

In [0]:
%sql
DESCRIBE history tpc_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-29T14:25:10.000Z,149468723871717,balaji.raghu@one-consultants.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> Training copy of samples.tpch.orders, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""0b1ff4ce-244b-4968-a931-7736b70d9b80"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-15625843-a367-4b96-86f2-67fd61795b3e"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-cb0a575e-3db8-4895-b768-d1e93e1a786e""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(986389444892964),3e4d4f86-1b17-4c03-a013-8982287a748c,1029-141424-aoz5m7d9-v2n,null,WriteSerializable,true,"Map(numFiles -> 3, numOutputRows -> 6831899, numOutputBytes -> 171724193)",null,Databricks-Runtime/19.13.x-photon-scala2.13


In [0]:
%sql
INSERT INTO tpc_orders
SELECT * FROM samples.tpch.orders
WHERE year(o_orderdate) = 1998;

num_affected_rows,num_inserted_rows
668101,668101


In [0]:
%sql describe detail tpc_orders

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1a157365-f4d8-4b9f-a1fd-7abdc2e43a58,training.bronze.tpc_orders,Training copy of samples.tpch.orders,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c,2026-10-30T10:28:18.705Z,2026-10-30T10:32:34.000Z,List(),List(),6,187257552,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7c028fbc-f265-45fa-a68b-25fe53a9548c, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-e1e3df7b-c035-45eb-868d-cae14efa9422, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-04f9a036-7135-40bd-8c77-8abf691b39d3)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


## 2. UPDATE, DELETE and MERGE (Immutablity)
Every DML statement is an ACID transaction that creates a **new table version**. Delta never edits files in place: it writes new files (or deletion vectors) and records in the transaction log which files are added and removed.

In [0]:
%sql
-- Escalate large pending orders to URGENT
UPDATE tpc_orders
SET o_orderpriority = '1-URGENT'
WHERE o_orderstatus = 'P' AND o_totalprice > 300000;

num_affected_rows
15873


In [0]:
%sql
-- Remove fulfilled orders from Q1 1992
DELETE FROM tpc_orders
WHERE o_orderstatus = 'F' AND o_orderdate < '1992-04-01';

num_affected_rows
283095


In [0]:
%sql
-- Source: recent orders (exist -> UPDATE) + the first week of 1992 fulfilled orders (deleted above -> INSERT)
MERGE INTO tpc_orders AS t
USING (
  SELECT * FROM samples.tpch.orders WHERE o_orderdate >= '1998-07-15'
  UNION ALL
  SELECT * FROM samples.tpch.orders
  WHERE o_orderstatus = 'F' AND o_orderdate BETWEEN '1992-01-01' AND '1992-01-07'
) AS s
ON t.o_orderkey = s.o_orderkey
WHEN MATCHED THEN UPDATE SET t.o_comment = 'updated by merge'
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
81208,59340,0,21868


### How `SELECT * FROM tpc_orders` Works Internally (with Multiple Versions)

<img src="/Volumes/training/bronze/images/Screenshot 2026-10-20 at 13.26.01.png" width="800"/>
When you run a simple `SELECT * FROM tpc_orders`, Delta Lake performs several steps **before** any data is read:


```
SELECT * FROM tpc_orders
        │
        ▼
┌─────────────────────────────┐
│  1. Find _delta_log/        │
│  2. Find latest version     │
│     (highest NNN.json = v3) │
│  3. Replay v0.json →v3.json │
│     (add / remove actions)  │
│  4. Build active file list   │
│     (add – remove = current)│
│  5. Skip files via min/max   │
│     stats (if WHERE)        │
│  6. Read active Parquet files│
│     + apply deletion vectors │
│  7. Return results ✅        │
└─────────────────────────────┘
```

#### Key Takeaway
Even though the table folder may contain **many Parquet files from all past versions** (old files from before UPDATEs, DELETEs, and MERGEs), the transaction log ensures that only the files belonging to the **current version** are read. Old files remain on disk solely to support **time travel** — until `VACUUM` permanently removes them.

## 3. Storage Layout, Delta Log and Checkpoints
### 3.1 Storage layout and versions
A Delta table folder looks like this:



* Each commit JSON lists `add` / `remove` file actions plus metadata.
* Old Parquet files stay in storage after UPDATE/DELETE — that is what makes time travel possible — until `VACUUM` removes them.
* With **deletion vectors** (default on serverless), UPDATE/DELETE mark rows as deleted in a small `.bin` file instead of rewriting whole Parquet files.



In [0]:
%sql
-- One row per version: operation, parameters and operationMetrics (files added/removed, rows affected)
DESCRIBE HISTORY tpc_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-10-30T10:40:00.000Z,149468723871717,balaji.raghu@one-consultants.com,MERGE,"Map(predicate -> [""(o_orderkey#12438L = o_orderkey#12447L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(986389444892964),5fea51e4-a27c-4168-9f44-fcba5c6faa2f,1030-095212-b631cnfo-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 8, numTargetBytesAdded -> 1501143, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 3, numTargetRowsMatchedUpdated -> 59340, executionTimeMs -> 5308, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 21868, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1779, numTargetRowsUpdated -> 59340, numOutputRows -> 81208, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 81208, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3477)",null,Databricks-Runtime/19.13.x-photon-scala2.13
3,2026-10-30T10:37:50.000Z,149468723871717,balaji.raghu@one-consultants.com,DELETE,"Map(predicate -> [""((o_orderstatus#12150 = F) AND (o_orderdate#12152 < 1992-04-01))""])",null,List(986389444892964),54876c05-837f-4d39-97f9-cd7a29fdb8d8,1030-095212-b631cnfo-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 3, numDeletionVectorsRemoved -> 3, numAddedChangeFiles -> 0, executionTimeMs -> 1294, numDeletionVectorsUpdated -> 3, numDeletedRows -> 283095, scanTimeMs -> 734, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 556)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-10-30T10:37:21.000Z,149468723871717,balaji.raghu@one-consultants.com,UPDATE,"Map(predicate -> [""((o_orderstatus#11430 = P) AND (o_totalprice#11431 > 300000.00))""])",null,List(986389444892964),9213155e-0092-4118-94f2-acab04897408,1030-095212-b631cnfo-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 3, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4756, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1760, numAddedFiles -> 3, numUpdatedRows -> 15873, numAddedBytes -> 446078, rewriteTimeMs -> 2972)",null,Databricks-Runtime/19.13.x-photon-scala2.13
1,2026-10-30T10:32:34.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),733ffb42-2a11-4c7a-a7be-fa9c0a40a619,1030-095212-b631cnfo-v2n,0,WriteSerializable,true,"Map(numFiles -> 3, numOutputRows -> 668101, numOutputBytes -> 15533359)",null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-10-30T10:28:25.000Z,149468723871717,balaji.raghu@one-consultants.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> Training copy of samples.tpch.orders, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7c028fbc-f265-45fa-a68b-25fe53a9548c"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e1e3df7b-c035-45eb-868d-cae14efa9422""

In [0]:
%sql
-- Compare numFiles / sizeInBytes with version 0
DESCRIBE DETAIL tpc_orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1a157365-f4d8-4b9f-a1fd-7abdc2e43a58,training.bronze.tpc_orders,Training copy of samples.tpch.orders,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c,2026-10-30T10:28:18.705Z,2026-10-30T10:40:00.000Z,List(),List(),17,189204773,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7c028fbc-f265-45fa-a68b-25fe53a9548c, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-e1e3df7b-c035-45eb-868d-cae14efa9422, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-04f9a036-7135-40bd-8c77-8abf691b39d3)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 358308, numDeletionVectors -> 6)",true


In [0]:
%sql
-- Compare numFiles / sizeInBytes with version 0
DESCRIBE EXTENDED tpc_orders;

col_name,data_type,comment
o_orderkey,bigint,null
o_custkey,bigint,null
o_orderstatus,string,null
o_totalprice,"decimal(18,2)",null
o_orderdate,date,null
o_orderpriority,string,null
o_clerk,string,null
o_shippriority,int,null
o_comment,string,null
,,


In [0]:
%sql
-- How many Parquet files make up each version's snapshot
SELECT 0 AS version, 'CTAS'   AS operation, COUNT(DISTINCT _metadata.file_path) AS data_files, COUNT(*) AS row_count FROM tpc_orders VERSION AS OF 0
UNION ALL SELECT 1, 'INSERT', COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 1
UNION ALL SELECT 2, 'UPDATE', COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 2
UNION ALL SELECT 3, 'DELETE', COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 3
UNION ALL SELECT 4, 'MERGE',  COUNT(DISTINCT _metadata.file_path), COUNT(*) FROM tpc_orders VERSION AS OF 4
ORDER BY version;

version,operation,data_files,row_count
0,CTAS,3,6831899
1,INSERT,6,7500000
2,UPDATE,9,7500000
3,DELETE,9,7216905
4,MERGE,17,7238773


In [0]:
%sql
-- Files present in v1 but not in v0 = the new Parquet files written by INSERT
WITH v0 AS (SELECT DISTINCT _metadata.file_name AS f FROM tpc_orders VERSION AS OF 0),
     v1 AS (SELECT _metadata.file_name AS f, _metadata.file_size AS size_bytes, COUNT(*) AS row_count
            FROM tpc_orders VERSION AS OF 1 GROUP BY ALL)
SELECT v1.* FROM v1 LEFT ANTI JOIN v0 USING (f);

f,size_bytes,row_count
part-00000-ee75b061-d724-4cd7-9f19-1977a0fea84f.c000.zstd.parquet,6998374,301530
part-00001-a30146b0-36af-4099-94ce-2c02ef704a6a.c000.zstd.parquet,7018072,301868
part-00002-5adc3f64-eb1a-4fff-869f-376724ed3bef.c000.zstd.parquet,1516913,64703


In [0]:
%sql
-- Physical files backing the current version (full storage path, size, rows)
SELECT _metadata.file_path AS file_path,
       _metadata.file_size AS size_bytes,
       COUNT(*)            AS row_count
FROM tpc_orders
GROUP BY ALL
ORDER BY file_path;

file_path,size_bytes,row_count
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00000-4a032275-9beb-4e6f-9439-5a5ed3dd19f4.c000.zstd.parquet,201112,7227
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00000-ee75b061-d724-4cd7-9f19-1977a0fea84f.c000.zstd.parquet,6998374,275154
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00000-f6c06262-308b-4dfb-9e86-a3321ea69206.c000.zstd.parquet,207468,13484
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00000-f87ae8d0-5a97-4159-a9f6-124b41e08c90.c000.zstd.parquet,77557276,2951002
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00001-1a69e953-631d-47cf-bb33-a68b82ccf45a.c000.zstd.parquet,210538,13629
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00001-661d2970-d8b3-44be-82c2-a7be4fbba4b1.c000.zstd.parquet,77547434,2950371
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00001-a30146b0-36af-4099-94ce-2c02ef704a6a.c000.zstd.parquet,7018072,274755
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00001-a3c271be-320f-4cc3-8e3a-6428b5b3eaee.c000.zstd.parquet,198278,7130
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00002-1b498063-e975-4826-8121-c2aa8153fab1.c000.zstd.parquet,16619483,631558
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c/part-00002-5adc3f64-eb1a-4fff-869f-376724ed3bef.c000.zstd.parquet,1516913,58852


### 3.2 Checkpoints in the Delta Log
Replaying every JSON commit from version 0 becomes expensive as versions grow. To solve this, Delta Lake writes a **checkpoint** file — a compact Parquet snapshot of the entire table state at a given version.

**What a checkpoint contains:**
- All currently active `add` file actions (the complete set of live Parquet data files)
- Table metadata (schema, partition columns, table properties)


**How checkpoints speed up reads:**
- Instead of replaying `v0.json → v0.json → … → vN.json`, a reader loads the **latest checkpoint** and replays only the JSON commits *after* it.
- Example: if a checkpoint exists at version 10, reading version 12 means loading `checkpoint 10` + replaying `11.json` + `12.json` — just 3 files instead of 13.

**When checkpoints are written:**
- By default, every **10** commits (`delta.checkpointInterval` table property).
- Databricks may also write checkpoints adaptively based on log size.
- You can customise the interval: `ALTER TABLE t SET TBLPROPERTIES ('delta.checkpointInterval' = '5');`

**The `_last_checkpoint` file:**
- A small JSON file in `_delta_log/` that tells readers which checkpoint is the newest, so they don't have to list the directory.



In [0]:
%sql
-- Tiny table; checkpoint every 5 commits instead of the default 10
CREATE TABLE IF NOT EXISTS tpc_log_demo (id INT, note STRING)
TBLPROPERTIES ('delta.checkpointInterval' = '5');

In [0]:
%sql
-- Each INSERT is one commit -> one new JSON file in _delta_log
BEGIN
  DECLARE i INT DEFAULT 1;
  WHILE i <= 12 DO
    INSERT INTO tpc_log_demo VALUES (i, concat('commit ', i));
    SET i = i + 1;
  END WHILE;
END;

In [0]:
%sql
-- One row per commit JSON file; isBlindAppend / operationMetrics come from commitInfo
DESCRIBE HISTORY tpc_log_demo;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
12,2026-10-30T12:38:24.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,11,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1040)",null,Databricks-Runtime/19.13.x-photon-scala2.13
11,2026-10-30T12:38:23.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,10,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1040)",null,Databricks-Runtime/19.13.x-photon-scala2.13
10,2026-10-30T12:38:19.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,9,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1040)",null,Databricks-Runtime/19.13.x-photon-scala2.13
9,2026-10-30T12:38:18.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,8,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1035)",null,Databricks-Runtime/19.13.x-photon-scala2.13
8,2026-10-30T12:38:17.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1035)",null,Databricks-Runtime/19.13.x-photon-scala2.13
7,2026-10-30T12:38:16.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1035)",null,Databricks-Runtime/19.13.x-photon-scala2.13
6,2026-10-30T12:38:15.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1035)",null,Databricks-Runtime/19.13.x-photon-scala2.13
5,2026-10-30T12:38:10.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1035)",null,Databricks-Runtime/19.13.x-photon-scala2.13
4,2026-10-30T12:38:09.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> false, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),aa9616fd-40ac-4c9a-8955-f3220ed61d60,1030-095212-b631cnfo-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1035)",null,Databricks-Runtime/19.13.x-photon-scala

In [0]:
%sql
SHOW TBLPROPERTIES tpc_log_demo;

key,value
clusterByAuto,true
clusteringColumns,[]
delta.checkpointInterval,5
delta.checkpointPolicy,v2
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.clustering,supported
delta.feature.deletionVectors,supported
delta.feature.domainMetadata,supported


Check the storage location https://portal.azure.com/#@one-consultants.com/resource/subscriptions/11ebd399-8f6c-46af-b26f-71fc4572151b/resourceGroups/ONE-Azure/providers/Microsoft.Storage/storageAccounts/unityms/storagebrowser

## 4. Time Travel
Query any earlier version with `VERSION AS OF n` or `TIMESTAMP AS OF '<ts>'` (use a timestamp from `DESCRIBE HISTORY`). `RESTORE TABLE ... TO VERSION AS OF n` rolls a table back by writing a new version.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM tpc_orders VERSION AS OF 0) AS v0_rows,
  (SELECT COUNT(*) FROM tpc_orders VERSION AS OF 1) AS v1_rows,
  (SELECT COUNT(*) FROM tpc_orders VERSION AS OF 3) AS v3_rows_after_delete,
  (SELECT COUNT(*) FROM tpc_orders)                 AS current_rows;

v0_rows,v1_rows,v3_rows_after_delete,current_rows
6831899,7500000,7216905,7238773


In [0]:
%sql
-- Rows whose priority changed between v1 (before UPDATE) and v2 (after UPDATE)
SELECT b.o_orderkey, b.o_totalprice,
       b.o_orderpriority AS priority_v1,
       a.o_orderpriority AS priority_v2
FROM tpc_orders VERSION AS OF 1 b
JOIN tpc_orders VERSION AS OF 2 a USING (o_orderkey)
WHERE b.o_orderpriority <> a.o_orderpriority
LIMIT 20;

o_orderkey,o_totalprice,priority_v1,priority_v2
8535398,315058.14,4-NOT SPECIFIED,1-URGENT
8664615,303556.87,5-LOW,1-URGENT
22988422,303111.59,5-LOW,1-URGENT
18243495,304065.75,5-LOW,1-URGENT
8655907,332610.34,2-HIGH,1-URGENT
8706208,317512.28,3-MEDIUM,1-URGENT
23791142,349917.65,2-HIGH,1-URGENT
22964579,322760.78,4-NOT SPECIFIED,1-URGENT
8930119,322132.12,3-MEDIUM,1-URGENT
19131206,317562.30,2-HIGH,1-URGENT


In [0]:
%sql
-- Replace the timestamp with one from DESCRIBE HISTORY, then uncomment:
-- SELECT COUNT(*) FROM tpc_orders TIMESTAMP AS OF '2026-10-16T10:00:00';
-- RESTORE TABLE tpc_orders TO VERSION AS OF 1;

-- Side-by-side: the same orders in v1 vs the current version (after MERGE)
SELECT 'v1' AS version, o_orderkey, o_orderstatus, o_comment FROM tpc_orders VERSION AS OF 1 WHERE o_orderdate = '1998-08-02'
UNION ALL
SELECT 'current', o_orderkey, o_orderstatus, o_comment FROM tpc_orders WHERE o_orderdate = '1998-08-02'
ORDER BY o_orderkey, version
LIMIT 10;

version,o_orderkey,o_orderstatus,o_comment
current,4678,O,updated by merge
current,4678,O,updated by merge
current,4678,O,updated by merge
current,4678,O,updated by merge
v1,4678,O,side of the bold platelets detect slyly blithely ironic e
current,7969,O,updated by merge
current,7969,O,updated by merge
current,7969,O,updated by merge
current,7969,O,updated by merge
v1,7969,O,uriously regular instructions. slyly fin


In [0]:
%sql
-- table@vN is shorthand for VERSION AS OF N
SELECT 'v0' AS version, COUNT(*) AS row_count FROM tpc_orders@v0
UNION ALL
SELECT 'v3', COUNT(*) FROM tpc_orders@v3;

version,row_count
v0,6831899
v3,7216905


**How far back can you travel?** A version is readable only while both its commit JSON (kept for `delta.logRetentionDuration`, default 30 days) and its data files (kept until VACUUM removes them after `delta.deletedFileRetentionDuration`, default 7 days) still exist. `RESTORE` is demonstrated on the CDF table in the next section.

## 5. Change Data Feed (CDF)
With `delta.enableChangeDataFeed = true`, Delta records **row-level changes** for every commit. Downstream jobs can read only what changed instead of rescanning the table.

`table_changes('table', start_version [, end_version])` returns the changed rows plus three metadata columns:

| Column | Values |
|---|---|
| `_change_type` | `insert`, `delete`, `update_preimage`, `update_postimage` |
| `_commit_version` | Version that produced the change |
| `_commit_timestamp` | Commit time |

Change data is stored in `_change_data/` (or derived from deletion vectors / row tracking) and is subject to the same VACUUM retention as data files.

In [0]:
%sql
-- Version 0: July 1998 orders, CDF enabled at creation
CREATE TABLE IF NOT EXISTS tpc_orders_cdf
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
AS
SELECT * FROM samples.tpch.orders
WHERE o_orderdate >= '1998-07-01';

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- v1: UPDATE, v2: DELETE, v3: INSERT
UPDATE tpc_orders_cdf SET o_orderstatus = 'F' WHERE o_orderdate = '1998-07-01';
DELETE FROM tpc_orders_cdf WHERE o_orderdate = '1998-07-02';
INSERT INTO tpc_orders_cdf SELECT * FROM samples.tpch.orders WHERE o_orderdate = '1998-06-30';
DESCRIBE HISTORY tpc_orders_cdf;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-10-30T12:56:46.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),46b3f7ec-2f03-4245-a010-1e007a04ac66,1030-095212-b631cnfo-v2n,2,WriteSerializable,true,"Map(numFiles -> 3, numOutputRows -> 3064, numOutputBytes -> 92179)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-10-30T12:56:44.000Z,149468723871717,balaji.raghu@one-consultants.com,DELETE,"Map(predicate -> [""(o_orderdate#19633 = 1998-07-02)""])",null,List(986389444892964),a8851165-1c4c-4fb6-a766-13537126ab9b,1030-095212-b631cnfo-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 3, numDeletionVectorsRemoved -> 3, numAddedChangeFiles -> 0, executionTimeMs -> 850, numDeletionVectorsUpdated -> 3, numDeletedRows -> 3128, scanTimeMs -> 427, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 423)",null,Databricks-Runtime/19.13.x-photon-scala2.13
1,2026-10-30T12:56:42.000Z,149468723871717,balaji.raghu@one-consultants.com,UPDATE,"Map(predicate -> [""(o_orderdate#18350 = 1998-07-01)""])",null,List(986389444892964),8dc54678-a69c-42d7-9706-3b7a9829f793,1030-095212-b631cnfo-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 3, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 3, executionTimeMs -> 2604, numDeletionVectorsUpdated -> 0, scanTimeMs -> 529, numAddedFiles -> 3, numUpdatedRows -> 3127, numAddedBytes -> 99794, rewriteTimeMs -> 2074)",null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-10-30T12:56:32.000Z,149468723871717,balaji.raghu@one-consultants.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""82e0cdff-7097-4c8a-b6cf-972a95c37865"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ab42e523-86c3-47fa-9d37-13460b6fe24b"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7bf096c7-74cc-498b-a996-c83bb95cf232"",""delta.enableChangeDataFeed"":""true"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(986389444892964),8f558db8-aef1-4099-ae40-5eb455b90f01,1030-095212-b631cnfo-v2n,null,WriteSerializable,true,"Map(numFiles -> 3, numOutputRows -> 102941, numOutputBytes -> 2635341)",null,Databricks-Runtime/19.13.x-photon-scala2.13


In [0]:
%sql
-- CDF change summary by version and change type
SELECT _commit_version, _change_type, COUNT(*) AS rows_changed
FROM table_changes('tpc_orders_cdf', 0, 3)
GROUP BY ALL
ORDER BY _commit_version, _change_type;

-- WHERE IS CDF DATA STORED?
-- CDF data lives in a dedicated `_change_data/` subfolder inside the table's storage directory,
-- NOT in the main data files and NOT in the Delta Log (_delta_log/) JSON/checkpoint files.
--
-- Storage layout:
--   <table_path>/
--   ├── _delta_log/          ← commit JSON + checkpoints (metadata only, NO change rows)
--   ├── part-00000-...parquet ← regular data files (current table state)
--   └── _change_data/        ← CDF Parquet files with _change_type, _commit_version, _commit_timestamp
--
-- These _change_data files are subject to VACUUM just like regular data files,
-- governed by delta.deletedFileRetentionDuration (default 7 days).

_commit_version,_change_type,rows_changed
0,insert,102941
1,update_postimage,3127
1,update_preimage,3127
2,delete,3128
3,insert,3064


In [0]:
%sql
SELECT _commit_version, _change_type, *
FROM table_changes('tpc_orders_cdf', 0, 3) where o_orderdate = '1998-07-02'


_commit_version,_change_type,o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment,_change_type,_commit_version,_commit_timestamp
0,insert,11404295,219991,O,88886.31,1998-07-02,1-URGENT,Clerk#000001706,0,y ironic epitaphs sleep.,insert,0,2026-10-30T12:56:32.000Z
0,insert,11406628,649765,O,23514.32,1998-07-02,1-URGENT,Clerk#000003637,0,ajole furiously furiously even asymptotes! thin fr,insert,0,2026-10-30T12:56:32.000Z
0,insert,11421250,736987,O,97260.76,1998-07-02,4-NOT SPECIFIED,Clerk#000001698,0,ges haggle carefully according to the,insert,0,2026-10-30T12:56:32.000Z
0,insert,11424739,673970,O,35526.50,1998-07-02,5-LOW,Clerk#000000042,0,ully express packages nod quickly,insert,0,2026-10-30T12:56:32.000Z
0,insert,11429730,478087,O,168720.74,1998-07-02,3-MEDIUM,Clerk#000004425,0,even ideas nag slyly bo,insert,0,2026-10-30T12:56:32.000Z
0,insert,11434823,530326,O,155946.36,1998-07-02,4-NOT SPECIFIED,Clerk#000000479,0,regular epitaphs integrate fluffily. regular,insert,0,2026-10-30T12:56:32.000Z
0,insert,11434855,231505,O,178435.34,1998-07-02,3-MEDIUM,Clerk#000002171,0,ic requests x-ray furiously about th,insert,0,2026-10-30T12:56:32.000Z
0,insert,11436645,577867,O,224094.36,1998-07-02,4-NOT SPECIFIED,Clerk#000000428,0,ly special dependencies above the deposits ought to,insert,0,2026-10-30T12:56:32.000Z
0,insert,11456103,156622,O,159349.52,1998-07-02,2-HIGH,Clerk#000000094,0,arefully bold instructions: furiously ironic accounts haggle,insert,0,2026-10-30T12:56:32.000Z
0,insert,11458784,95152,O,88430.06,1998-07-02,2-HIGH,Clerk#000001344,0,ounts wake fluffily sometimes express dep,insert,0,2026-10-30T12:56:32.000Z


In [0]:
%sql
-- Each UPDATE produces a before (preimage) and after (postimage) row
SELECT o_orderkey, o_orderstatus, _change_type, _commit_version, _commit_timestamp
FROM table_changes('tpc_orders_cdf', 10, 11)
ORDER BY o_orderkey, _change_type DESC
LIMIT 10;

o_orderkey,o_orderstatus,_change_type,_commit_version,_commit_timestamp
8739,O,update_preimage,10,2026-10-23T13:32:18.000Z
8739,F,update_postimage,10,2026-10-23T13:32:18.000Z
12391,O,update_preimage,10,2026-10-23T13:32:18.000Z
12391,F,update_postimage,10,2026-10-23T13:32:18.000Z
12838,O,update_preimage,10,2026-10-23T13:32:18.000Z
12838,F,update_postimage,10,2026-10-23T13:32:18.000Z
17537,O,delete,11,2026-10-23T13:32:20.000Z
23779,O,update_preimage,10,2026-10-23T13:32:18.000Z
23779,F,update_postimage,10,2026-10-23T13:32:18.000Z
26048,O,delete,11,2026-10-23T13:32:20.000Z


In [0]:
%sql
-- Typical downstream pattern: keep only the last change per key (ready for MERGE into a target)
SELECT * EXCEPT (rn)
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY o_orderkey ORDER BY _commit_version DESC) AS rn
  FROM table_changes('tpc_orders_cdf', 11,13)
  WHERE _change_type <> 'update_preimage'
)
WHERE rn = 1
LIMIT 10;

o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment,_change_type,_commit_version,_commit_timestamp
8739,194687,F,111794.24,1998-07-01,2-HIGH,Clerk#000001906,0,tions boost about the furiously pending requests. quickl,update_postimage,13,2026-10-23T13:36:24.000Z
11268,20980,O,164458.97,1998-06-30,1-URGENT,Clerk#000000650,0,ptotes are blithely slyly silen,insert,12,2026-10-23T13:32:23.000Z
12391,499783,F,93052.82,1998-07-01,5-LOW,Clerk#000002164,0,cies are blithely. furious,update_postimage,13,2026-10-23T13:36:24.000Z
12838,55735,F,81465.36,1998-07-01,1-URGENT,Clerk#000002120,0,fully ironic foxes. quickly pending accounts about the fin,update_postimage,13,2026-10-23T13:36:24.000Z
17537,166621,O,231819.75,1998-07-02,5-LOW,Clerk#000000918,0,heodolites doze blithely theodolites. final instructions use q,delete,11,2026-10-23T13:32:20.000Z
23425,667327,O,49225.32,1998-06-30,4-NOT SPECIFIED,Clerk#000002315,0,usly express pinto beans across th,insert,12,2026-10-23T13:32:23.000Z
23779,299875,F,258095.91,1998-07-01,5-LOW,Clerk#000004931,0,ructions around the regular sau,update_postimage,13,2026-10-23T13:36:24.000Z
26048,125200,O,53767.63,1998-07-02,5-LOW,Clerk#000002305,0,"es. quickly ironic deposits wake sometimes against the express, pending d",delete,11,2026-10-23T13:32:20.000Z
26530,643411,F,21976.97,1998-07-01,4-NOT SPECIFIED,Clerk#000001678,0,s sleep carefully quickly thin asymptotes. blithely unusual theodol,update_postimage,13,2026-10-23T13:36:24.000Z
27041,480142,O,19667.86,1998-06-30,1-URGENT,Clerk#000001027,0,unusual dolphins. slyly regular ideas after the even theodolites x-ray s,insert,12,2026-10-23T13:32:23.000Z


In [0]:
%sql
-- Roll the CDF table back to its original load; RESTORE writes a NEW version (history is kept)
RESTORE TABLE tpc_orders_cdf TO VERSION AS OF 0;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
2635341,3,9,3,2827314,2635341


## Managed vs External Tables

**Managed table**
* Unity Catalog manages both the metadata **and** the data files.
* Data is stored in the managed storage location of the catalog / schema (e.g. `abfss://training@unityms.dfs.core.windows.net/...__unitystorage/...`).
* `DROP TABLE` removes the metadata **and** the data files (recoverable with `UNDROP` within 7 days).

**External table**
* Unity Catalog manages only the metadata; the data files stay in a path **you** control.
* Data is stored in a registered external location (ADLS path).
* Created with a `LOCATION` clause: `CREATE TABLE t (...) LOCATION 'abfss://<container>@<account>.dfs.core.windows.net/<path>'`.
* `DROP TABLE` removes only the metadata; the files remain in storage.
* Use when data must be shared with other tools / systems or already exists in the lake.

| | Managed | External |
|---|---|---|
| Metadata | Unity Catalog | Unity Catalog |
| Data location | Catalog / schema managed storage | Your external location |
| `LOCATION` clause | Not used | Required |
| `DROP TABLE` | Deletes metadata + data | Deletes metadata only |
| Predictive optimization | Yes | No |

All working tables in this notebook (`tpc_orders`, `tpc_orders_cdf`, ...) are **managed** tables. Check with `DESCRIBE EXTENDED <table>` (row `Type`)

#Syntax: CREATE TABLE 'TABLE_NAME' (...) LOCATION 'storage location path'.

#Practial Scenario:
* The data is already maintained in some other datalake before databricks. In Scania we have a scecario where our datafiles are present outside the our cloud storage(on cloud data lake), here we should not have control over droping the table data when table is dropped
* if the organization needs strict decoupling of storage and processing 

#Important thing to remember 
Schema can also be created with location keyword, in such case the tables created underthis (without location keyword) will still be treated as managed table, only when you create a table with keyword location it will be treated as external table


## 6. Constraints
| Type | Enforced? | Purpose |
|---|---|---|
| `NOT NULL` | Yes | Reject NULLs in a column |
| `CHECK (expr)` | Yes | Reject rows where the expression is false; stored as table property `delta.constraints.<name>` |
| `PRIMARY KEY` / `FOREIGN KEY` | No (informational) | Document relationships for tools and BI; with `RELY` the optimizer may use them to remove joins / aggregations |

A write that violates an enforced constraint fails as a whole — no partial data is committed.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_customer AS
SELECT * FROM samples.tpch.customer;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Re-runnable: drop constraints first if they already exist
ALTER TABLE tpc_orders_cdf DROP CONSTRAINT IF EXISTS fk_orders_customer;
ALTER TABLE tpc_customer   DROP CONSTRAINT IF EXISTS pk_customer;

ALTER TABLE tpc_customer ALTER COLUMN c_custkey SET NOT NULL;
ALTER TABLE tpc_customer ADD CONSTRAINT pk_customer PRIMARY KEY (c_custkey) RELY;

ALTER TABLE tpc_orders_cdf ALTER COLUMN o_orderkey SET NOT NULL;
ALTER TABLE tpc_orders_cdf ADD CONSTRAINT fk_orders_customer
  FOREIGN KEY (o_custkey) REFERENCES tpc_customer (c_custkey);

SELECT table_name, constraint_name, constraint_type
FROM information_schema.table_constraints
WHERE table_schema = 'bronze' AND table_name IN ('tpc_customer', 'tpc_orders_cdf');

table_name,constraint_name,constraint_type
tpc_orders_cdf,fk_orders_customer,FOREIGN KEY
tpc_customer,pk_customer,PRIMARY KEY


In [0]:
%sql
ALTER TABLE tpc_orders_cdf DROP CONSTRAINT IF EXISTS chk_price;
ALTER TABLE tpc_orders_cdf DROP CONSTRAINT IF EXISTS chk_status;

-- Existing rows are validated when a CHECK constraint is added
ALTER TABLE tpc_orders_cdf ADD CONSTRAINT chk_price  CHECK (o_totalprice > 0);
ALTER TABLE tpc_orders_cdf ADD CONSTRAINT chk_status CHECK (o_orderstatus IN ('F', 'O', 'P'));

-- CHECK constraints are stored as delta.constraints.* table properties
SHOW TBLPROPERTIES tpc_orders_cdf;

key,value
clusterByAuto,true
clusteringColumns,[]
delta.checkpointPolicy,v2
delta.constraints.chk_price,o_totalprice > 0
delta.constraints.chk_status,"o_orderstatus IN ('F', 'O', 'P')"
delta.enableChangeDataFeed,true
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.changeDataFeed,supported


In [0]:
%sql
-- Try to insert an invalid row (negative price, unknown status); the error is caught and displayed
BEGIN
  DECLARE EXIT HANDLER FOR SQLEXCEPTION
  BEGIN
    DECLARE msg STRING;
    GET DIAGNOSTICS CONDITION 1 msg = MESSAGE_TEXT;
    SELECT 'INSERT rejected' AS result, msg AS error_message;
  END;
  INSERT INTO tpc_orders_cdf VALUES
    (99999999, 1, 'X', -10.00, DATE'1998-08-01', '1-URGENT', 'Clerk#000000001', 0, 'invalid row');
  SELECT 'INSERT succeeded' AS result, NULL AS error_message;
END;

result,error_message
INSERT rejected,[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint chk_price (o_totalprice > 0) violated by row with values: - o_totalprice : -10.00. SQLSTATE: 23001


In [0]:
%sql
-- PRIMARY KEY is informational only — duplicates are NOT rejected
INSERT INTO tpc_customer
SELECT * FROM tpc_customer WHERE c_custkey = 1;

-- Proof: c_custkey = 1 now appears twice
SELECT *
FROM tpc_customer
WHERE c_custkey = 1


c_custkey,c_name,c_address,c_nationkey,c_phone,c_acctbal,c_mktsegment,c_comment
1,Customer#000000001,"IVhzIApeRb ot,c,E",15,25-989-741-2988,711.56,BUILDING,"to the even, regular platelets. regular, ironic epitaphs nag e"
1,Customer#000000001,"IVhzIApeRb ot,c,E",15,25-989-741-2988,711.56,BUILDING,"to the even, regular platelets. regular, ironic epitaphs nag e"


## 7. Indexes and Data Skipping
 It avoids reading data using:

1. **File statistics (built-in index)** — min/max/null counts per file stored in each `add` action. By default collected for the first 32 columns (`delta.dataSkippingNumIndexedCols`) or for an explicit list (`delta.dataSkippingStatsColumns`). Filters skip files whose range cannot match.
2. **Layout techniques** — Z-order and Liquid Clustering make file statistics far more selective (section 8).
3. **Optimizer statistics** — `ANALYZE TABLE ... COMPUTE STATISTICS` gives the cost-based optimizer row counts and column distributions for join planning.

In [0]:
%sql
-- Collect file statistics only on columns used in filters, then recompute stats for existing files
ALTER TABLE tpc_orders_cdf SET TBLPROPERTIES (
  'delta.dataSkippingStatsColumns' = 'o_orderkey,o_custkey,o_orderdate,o_totalprice,o_orderstatus'
);
ANALYZE TABLE tpc_orders_cdf COMPUTE DELTA STATISTICS;

In [0]:
%sql
ANALYZE TABLE tpc_orders_cdf COMPUTE STATISTICS FOR ALL COLUMNS;
-- Column-level stats: min, max, distinct_count, num_nulls
DESCRIBE EXTENDED tpc_orders_cdf o_custkey;

info_name,info_value
col_name,o_custkey
data_type,bigint
comment,NULL
min,25
max,749998
num_nulls,0
distinct_count,92986
avg_col_len,8
max_col_len,8
histogram,NULL


## 8. Performance Techniques
Delta offers several complementary ways to make queries read less data and write well-sized files:
8.1 Partitioning, 8.2 Z-Ordering, 8.3 Liquid Clustering, 8.4 Compaction, 8.5 Optimized Writes & Auto Compaction, 8.6 Predictive Optimization, 8.7 Key Table Properties.

### 8.1 Partitioning
`PARTITIONED BY` physically splits data into one folder per value (`order_year=1995/part-...parquet`). Queries filtering on the partition column read only matching folders (**partition pruning**).

Use only for low-cardinality columns with large partitions (≥ 1 GB each). High-cardinality partitioning creates many small files — for most new tables prefer Liquid Clustering.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_orders_partitioned
PARTITIONED BY (order_year)
AS
SELECT *, year(o_orderdate) AS order_year
FROM samples.tpch.orders;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW PARTITIONS tpc_orders_partitioned;

order_year
1994
1992
1996
1997
1998
1995
1993


In [0]:
%sql
-- Note the order_year=YYYY folder in each file path
SELECT order_year,
       COUNT(DISTINCT _metadata.file_path) AS files,
       COUNT(*)                            AS row_count,
       MIN(_metadata.file_path)            AS sample_file_path
FROM tpc_orders_partitioned
GROUP BY order_year
ORDER BY order_year;

order_year,files,row_count,sample_file_path
1992,1,1139873,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1992/part-00002-5e3f441f-6a9d-4ea5-a07e-1a35251b8d87.c000.zstd.parquet
1993,1,1138893,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1993/part-00004-c193dd14-b7d2-4756-b8ba-ad61127dcc7c.c000.zstd.parquet
1994,1,1137944,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1994/part-00000-0e40181b-b74e-4cfb-84ce-b91016f40272.c000.zstd.parquet
1995,1,1136308,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1995/part-00005-86ee6d21-505c-4441-9055-3933219a14a9.c000.zstd.parquet
1996,1,1141556,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1996/part-00006-c18099c5-52bf-4e8e-a4aa-d3484dc7efde.c000.zstd.parquet
1997,1,1137325,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1997/part-00003-ad6e97e4-dfb2-48c9-9611-b6a01495de14.c000.zstd.parquet
1998,1,668101,abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859/order_year=1998/part-00001-f3e66885-475a-46ca-9924-cbca61a24d58.c000.zstd.parquet


In [0]:
%sql
-- Look for PartitionFilters: [order_year = 1995] in the scan node
EXPLAIN FORMATTED
SELECT o_orderpriority, COUNT(*), SUM(o_totalprice)
FROM tpc_orders_partitioned
WHERE order_year = 1995
GROUP BY o_orderpriority;

plan
"== Physical Plan == AdaptiveSparkPlan (6) +- == Initial Plan == PhotonResultStage (5) +- PhotonColumnarToRow (4) +- PhotonGroupingAgg (3) +- PhotonProject (2) +- PhotonScan parquet training.bronze.tpc_orders_partitioned (1) (1) PhotonScan parquet training.bronze.tpc_orders_partitioned Output [3]: [o_totalprice#30151, o_orderpriority#30153, order_year#30157] Location: PreparedDeltaFileIndex [abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/d5a0b365-1d6c-4b43-8874-e5cd6f186859] PartitionFilters: [isnotnull(order_year#30157), (order_year#30157 = 1995)] ReadSchema: struct (2) PhotonProject Input [3]: [o_totalprice#30151, o_orderpriority#30153, order_year#30157] Arguments: [o_totalprice#30151, o_orderpriority#30153] (3) PhotonGroupingAgg Input [2]: [o_totalprice#30151, o_orderpriority#30153] Arguments: [o_orderpriority#30153], [count(1), sum(o_totalprice#30151)], [count(1)#30158L, sum(o_totalprice)#30159], [o_orderpriority#30153, count(1)#30158L AS COUNT(*)#30160L, sum(o_totalprice)#30159 AS SUM(o_totalprice)#30161], true (4) PhotonColumnarToRow Input [3]: [o_orderpriority#30153, COUNT(*)#30160L, SUM(o_totalprice)#30161] (5) PhotonResultStage Input [3]: [o_orderpriority#30153, COUNT(*)#30160L, SUM(o_totalprice)#30161] (6) AdaptiveSparkPlan Output [3]: [o_orderpriority#30153, COUNT(*)#30160L, SUM(o_totalprice)#30161] Arguments: isFinalPlan=false == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = tpc_orders_partitioned"


### 8.2 Z-Ordering
`OPTIMIZE ... ZORDER BY (col)` compacts small files and co-locates similar values of `col` in the same files. Delta keeps min/max statistics per file, so a filter like `o_custkey = 12345` can skip files whose range does not contain the value (**data skipping**).

Below, compare each file's `o_custkey` range before and after Z-ordering: before, every file spans almost the full range; after, ranges are narrow.

In [0]:
%sql
SELECT _metadata.file_name AS file_name, COUNT(*) AS row_count,
       MIN(o_custkey) AS min_custkey, MAX(o_custkey) AS max_custkey
FROM tpc_orders
GROUP BY ALL
ORDER BY min_custkey;

file_name,row_count,min_custkey,max_custkey
part-00000-f87ae8d0-5a97-4159-a9f6-124b41e08c90.c000.zstd.parquet,2951002,1,749999
part-00001-661d2970-d8b3-44be-82c2-a7be4fbba4b1.c000.zstd.parquet,2950371,1,749999
part-00002-1b498063-e975-4826-8121-c2aa8153fab1.c000.zstd.parquet,631558,1,749999
part-00001-a30146b0-36af-4099-94ce-2c02ef704a6a.c000.zstd.parquet,274755,2,749999
part-00000-ee75b061-d724-4cd7-9f19-1977a0fea84f.c000.zstd.parquet,275154,2,749998
part-00005-ca4544ac-2094-4692-bf9a-098a1d4052f6.c000.zstd.parquet,9899,7,749983
part-00002-5adc3f64-eb1a-4fff-869f-376724ed3bef.c000.zstd.parquet,58852,13,749968
part-00000-f6c06262-308b-4dfb-9e86-a3321ea69206.c000.zstd.parquet,13484,25,749990
part-00002-e34f32bf-6e77-44e4-b625-07614f405e55.c000.zstd.parquet,13167,34,749992
part-00003-3a3d1529-7195-4649-a539-1afc6efd3610.c000.zstd.parquet,13209,34,749987


In [0]:
%sql
OPTIMIZE tpc_orders ZORDER BY (o_custkey);

path,metrics
abfss://training@unityms.dfs.core.windows.net/bronze/__unitystorage/schemas/94a6d5cf-a092-40c1-a6f5-a37657cd1dba/tables/7c028fbc-f265-45fa-a68b-25fe53a9548c,"List(2, 17, List(83498856, 87174065, 8.53364605E7, 2, 170672921), List(46688, 77557276, 1.1129692529411765E7, 17, 189204773), 0, List(minCubeSize(107374182400), List(0, 0), List(17, 189204773), 0, List(17, 189204773), 1, null), null, 0, 1, 17, 0, false, 0, 0, 1793367276794, 1793367286207, 8, 1, null, List(6, 358308), null, 9, 9, 8083, 0, null, null, 0)"


In [0]:
%sql
SELECT _metadata.file_name AS file_name, COUNT(*) AS row_count,
       MIN(o_custkey) AS min_custkey, MAX(o_custkey) AS max_custkey
FROM tpc_orders
GROUP BY ALL
ORDER BY min_custkey;

file_name,row_count,min_custkey,max_custkey
part-00000-7f9745f5-3549-4542-86e2-c919821419d2.c000.zstd.parquet,3538563,1,366568
part-00001-61c19329-77df-4cd6-92df-a8a6f367b90c.c000.zstd.parquet,3700210,366568,749999


### 8.3 Liquid Clustering
`CLUSTER BY` replaces both partitioning and Z-order for new tables:
* No folder-per-value; avoids small-file problems on high-cardinality keys.
* Clustering is incremental — `OPTIMIZE` only reclusters new data.
* Keys can be changed later with `ALTER TABLE ... CLUSTER BY` without rewriting the table.
* `CLUSTER BY AUTO` lets Databricks pick keys from the query workload (requires predictive optimization).

Not compatible with `PARTITIONED BY` or `ZORDER` on the same table.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS tpc_lineitems_clustered
CLUSTER BY (l_orderkey)
AS
SELECT * from samples.tpch_sf100.lineitem;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT _metadata.file_name AS file_name, COUNT(*) AS row_count,
       MIN(l_orderkey) AS min_l_orderkey, MAX(l_orderkey) AS max_l_orderkey
FROM tpc_lineitems_clustered
GROUP BY file_name
ORDER BY min_l_orderkey;

file_name,row_count,min_l_orderkey,max_l_orderkey
part-00143-2874e046-026c-4bdf-99c1-57c67b4c3da7.c000.zstd.parquet,2438494,1,2438529
part-00227-adaa8883-8944-4e8d-b900-db5c2c6699e3.c000.zstd.parquet,2506995,2438530,4944228
part-00150-6b057ab3-a256-4f5e-b7e4-ccbb048711a2.c000.zstd.parquet,2267465,4944229,7214560
part-00035-c22ae32b-5a5f-4d3d-80ef-089883780bd9.c000.zstd.parquet,2570049,7214561,9782790
part-00062-0b54935a-dc6b-4538-b164-c22883f947a3.c000.zstd.parquet,2547605,9782791,12333314
part-00146-7addd261-bc30-4058-9259-a268ece7ee6e.c000.zstd.parquet,2202687,12333315,14535970
part-00233-3bf62b16-f0d9-4d59-99f0-293c385c2f6a.c000.zstd.parquet,2812383,14535971,17348806
part-00011-1d144178-72ba-4fd7-9911-bd7dbbf53559.c000.zstd.parquet,2064893,17348807,19414823
part-00041-a3e3a646-3071-44ed-8263-a32321f2a8f8.c000.zstd.parquet,2241548,19414848,21657379
part-00034-50b4c4c1-cd92-46a8-98fc-3a5bd87ff111.c000.zstd.parquet,2532286,21657380,24187399


In [0]:
%sql explain select * from tpc_lineitems_clustered where l_orderkey in (9130853)

plan
"== Physical Plan == PhotonResultStage +- PhotonColumnarToRow +- PhotonScan parquet one_dbx_training.default.tpc_lineitems_clustered[l_orderkey#32112L,l_partkey#32113L,l_suppkey#32114L,l_linenumber#32115,l_quantity#32116,l_extendedprice#32117,l_discount#32118,l_tax#32119,l_returnflag#32120,l_linestatus#32121,l_shipdate#32122,l_commitdate#32123,l_receiptdate#32124,l_shipinstruct#32125,l_shipmode#32126,l_comment#32127] DataFilters: [isnotnull(l_orderkey#32112L), (l_orderkey#32112L = 9130853)], DictionaryFilters: [(l_orderkey#32112L = 9130853)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://unity-catalog-storage@dbstoragedusvtfzv2vuhk.dfs.core.win..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<l_orderkey:bigint,l_partkey:bigint,l_suppkey:bigint,l_linenumber:int,l_quantity:decimal(18..., RequiredDataFilters: [isnotnull(l_orderkey#32112L), (l_orderkey#32112L = 9130853)] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = tpc_lineitems_clustered"


In [0]:
%sql select * from tpc_lineitems_clustered where l_orderkey in (9130853,1899877)

--Check performance

l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment
1899877,16518701,18734,1,11.00,18907.68,0.03,0.04,A,F,1994-03-09,1994-03-23,1994-03-14,TAKE BACK RETURN,FOB,dencies are quickly above the ruthles
1899877,9794503,794504,2,16.00,25552.32,0.09,0.00,A,F,1994-05-08,1994-02-08,1994-05-15,NONE,MAIL,p unusual realms.
9130853,18431080,431081,6,12.00,12121.92,0.01,0.03,A,F,1994-12-10,1994-11-16,1994-12-14,DELIVER IN PERSON,SHIP,pitaphs ac
9130853,1440502,940505,2,28.00,40388.04,0.06,0.06,R,F,1994-12-13,1994-10-28,1994-12-18,TAKE BACK RETURN,MAIL,uests haggl
9130853,3900842,150846,1,38.00,70020.70,0.03,0.01,R,F,1994-10-12,1994-11-11,1994-11-02,NONE,MAIL,ithely spec
9130853,10510787,510788,3,50.00,89863.00,0.00,0.02,A,F,1994-10-15,1994-10-20,1994-10-29,COLLECT COD,RAIL,ep against the slyl
9130853,11156109,156110,7,22.00,25620.10,0.03,0.05,A,F,1994-10-21,1994-10-30,1994-10-23,COLLECT COD,RAIL,"ncies wake. pending, express depos"
9130853,4571812,571813,5,6.00,11301.54,0.00,0.06,R,F,1994-10-24,1994-11-26,1994-10-25,NONE,REG AIR,ar pinto beans. slyly ironic dolphins wa
9130853,14790538,40553,4,32.00,52089.60,0.02,0.07,R,F,1994-11-07,1994-11-13,1994-11-15,DELIVER IN PERSON,RAIL,ress theodoli


In [0]:
%sql

-- Let Databricks choose keys automatically
ALTER TABLE tpc_lineitems_clustered CLUSTER BY AUTO;
DESCRIBE HISTORY tpc_lineitems_clustered;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-30T13:43:39.000Z,149468723871717,balaji.raghu@one-consultants.com,CLUSTER BY,"Map(oldClusteringColumns -> l_orderkey, newClusteringColumns -> l_orderkey, clusterByAuto -> true)",null,List(986389444892964),005fcac7-6d51-4dc9-b1b2-a9ab8add65c0,1030-095212-b631cnfo-v2n,0,SnapshotIsolation,true,Map(),null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-10-30T13:41:00.000Z,149468723871717,balaji.raghu@one-consultants.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [""l_orderkey""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""9baa53cf-579f-4352-baf2-33f874064c5b"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a6aea231-1b49-4fdf-8b7e-3f1d4d31de4b"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-69ce7313-7cce-4ccd-9f42-7b288bb43a7d""}, statsOnLoad -> true, clusteringOnWriteStatus -> kdtree triggered)",null,List(986389444892964),2eb8322b-ac00-465d-9cb7-a9792c91cf5f,1030-095212-b631cnfo-v2n,null,WriteSerializable,true,"Map(numFiles -> 256, numOutputRows -> 600037902, numOutputBytes -> 16177054266)",null,Databricks-Runtime/19.13.x-photon-scala2.13


#### Comparison: Partitioning vs Z-Order vs Liquid Clustering
| Feature | Partitioning | Z-Order | Liquid Clustering |
|---|---|---|---|
| Physical layout | Folder per value | Sorted within files | Clustered files, no folders |
| Best for | Low-cardinality, very large tables | Filter columns on existing tables | Default choice for new tables |
| Change keys | Full rewrite | Re-run OPTIMIZE | `ALTER TABLE CLUSTER BY` |
| Incremental | n/a | No (rewrites) | Yes |
| Combine with others | With Z-order | With partitioning | Standalone only |

#### When to choose which

**Choose Liquid Clustering when** (recommended default for new tables)
* You are creating a new table, or can recreate an existing one.
* Filter columns are high-cardinality (e.g. `o_custkey`, `o_orderkey`) or you filter on several columns together (`o_custkey`, `o_orderdate`).
* Query patterns may change over time, so you want to change keys with `ALTER TABLE ... CLUSTER BY` without rewriting data.
* The table receives frequent incremental writes (streaming, MERGE, concurrent writers) and you want `OPTIMIZE` to recluster only new data.
* You do not know the best keys yet: use `CLUSTER BY AUTO` with predictive optimization.

**Choose Partitioning when**
* The table is very large (roughly 1 TB+) and each partition still holds at least about 1 GB of data.
* The column is low-cardinality and stable (e.g. date, year, region, country), never high-cardinality IDs.
* You need partition-level operations: dropping/overwriting whole partitions (`REPLACE WHERE`), data lifecycle by date, or physical isolation for governance.
* An existing pipeline or external reader depends on the folder layout.
* Avoid it on small or medium tables: over-partitioning creates many tiny files and slows queries.

**Choose Z-Order when**
* You have an existing (often partitioned) table that you cannot migrate to liquid clustering.
* Queries filter on 1 to 4 high-cardinality columns *within* partitions (e.g. partition by year, Z-order by `o_custkey`).
* Data is mostly written in batches and you can afford periodic full `OPTIMIZE ... ZORDER BY` rewrites.

**Quick decision guide**

| Situation | Recommendation |
|---|---|
| New table, any size | Liquid Clustering (`CLUSTER BY` or `CLUSTER BY AUTO`) |
| Filters on high-cardinality columns | Liquid Clustering (or Z-Order on legacy tables) |
| Huge table, low-cardinality date/region filter, partition-level deletes or overwrites | Partitioning (optionally + Z-Order) |
| Legacy partitioned table, slow filters on other columns | Add Z-Order, or migrate to Liquid Clustering |
| Table under ~1 TB | Do not partition; use Liquid Clustering |
| Unknown or changing query patterns | `CLUSTER BY AUTO` |

### 8.4 Compaction (bin-packing) and the small-file problem
Frequent small writes (streaming, many small batches, over-partitioned jobs) produce thousands of tiny files. Every file means an extra metadata entry, an extra cloud storage request and extra per-file overhead for queries.

`OPTIMIZE table` (without ZORDER) **bin-packs** small files into files of roughly `delta.targetFileSize` (auto-tuned, typically 128 MB–1 GB). It is idempotent, transactional, and only rewrites files that need it. Below we deliberately create 200 small files, then compact them.

In [0]:
%sql
-- Optimized writes / auto compaction OFF, and REPARTITION(200) forces ~200 output files
CREATE TABLE IF NOT EXISTS tpc_orders_small
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'false',
  'delta.autoOptimize.autoCompact'   = 'false'
)
AS
SELECT /*+ REPARTITION(200) */ *
FROM samples.tpch.orders
WHERE year(o_orderdate) = 1995;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*)                         AS num_files,
       ROUND(AVG(size_bytes) / 1048576, 2) AS avg_file_mb,
       ROUND(SUM(size_bytes) / 1048576, 1) AS total_mb
FROM (SELECT DISTINCT _metadata.file_path, _metadata.file_size AS size_bytes FROM tpc_orders_small);

num_files,avg_file_mb,total_mb
200,0.16,31.3


In [0]:
%sql
-- metrics.numFilesRemoved / numFilesAdded show the effect
OPTIMIZE tpc_orders_small;

path,metrics
abfss://unity-catalog-storage@dbstoragedusvtfzv2vuhk.dfs.core.windows.net/7405610410794404/__unitystorage/catalogs/56a27f63-cc02-48cc-98b8-e7ffa11bfee6/tables/7c9f753b-0d7e-46ea-8958-e3f9a7f5bf2a,"List(1, 200, List(28397912, 28397912, 2.8397912E7, 1, 28397912), List(161938, 165624, 164013.835, 200, 32802767), 0, null, null, 0, 1, 200, 0, true, 0, 0, 1793368921126, 1793368925299, 8, 1, null, List(0, 0), null, 9, 9, 1204, 0, null, null, 0)"


In [0]:
%sql
SELECT COUNT(*)                         AS num_files,
       ROUND(AVG(size_bytes) / 1048576, 2) AS avg_file_mb,
       ROUND(SUM(size_bytes) / 1048576, 1) AS total_mb
FROM (SELECT DISTINCT _metadata.file_path, _metadata.file_size AS size_bytes FROM tpc_orders_small);

num_files,avg_file_mb,total_mb
1,27.08,27.1


### 8.5 Optimized Writes and Auto Compaction
Rather than fixing small files afterwards, prevent them at write time:

| Property | What it does |
|---|---|
| `delta.autoOptimize.optimizeWrite = true` | **Optimized writes**: shuffles data before writing so each partition/task writes fewer, larger files (~128 MB). Small extra shuffle cost at write, much better reads. |
| `delta.autoOptimize.autoCompact = auto` | **Auto compaction**: after a write commits, synchronously compacts small files in the affected partitions/table when enough small files accumulate. |

Both are on by default for many operations on Databricks (MERGE, UPDATE, DELETE, CTAS, serverless). We now turn them on for `tpc_orders_small` and repeat the same 200-way write.

In [0]:
%sql
ALTER TABLE tpc_orders_small SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'auto'
);

In [0]:
%sql
INSERT INTO tpc_orders_small
SELECT /*+ REPARTITION(200) */ *
FROM samples.tpch.orders
WHERE year(o_orderdate) = 1996;

num_affected_rows,num_inserted_rows
1141556,1141556


In [0]:
%sql
SELECT COUNT(*)                         AS num_files,
       ROUND(AVG(size_bytes) / 1048576, 2) AS avg_file_mb,
       ROUND(SUM(size_bytes) / 1048576, 1) AS total_mb
FROM (SELECT DISTINCT _metadata.file_path, _metadata.file_size AS size_bytes FROM tpc_orders_small);

num_files,avg_file_mb,total_mb
2,27.0,54.0


In [0]:
%sql
-- Compare operationMetrics.numFiles of the CTAS (version 0) with the INSERT after optimized writes
DESCRIBE HISTORY tpc_orders_small;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-10-30T14:04:06.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),ef712025-5096-4c1a-b2ab-43ecfad79118,1030-095212-b631cnfo-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1141556, numOutputBytes -> 28226042)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-10-30T14:03:56.000Z,149468723871717,balaji.raghu@one-consultants.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""auto""})",null,List(986389444892964),bb7486a9-ba38-460e-83a7-d63837c4ee03,1030-095212-b631cnfo-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-photon-scala2.13
1,2026-10-30T14:02:04.000Z,149468723871717,balaji.raghu@one-consultants.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(986389444892964),7433fc7c-b834-4b97-b361-d41464fc2d58,1030-095212-b631cnfo-v2n,0,SnapshotIsolation,false,"Map(numRemovedFiles -> 200, numRemovedBytes -> 32802767, p25FileSize -> 28397912, numDeletionVectorsRemoved -> 0, minFileSize -> 28397912, numAddedFiles -> 1, maxFileSize -> 28397912, p75FileSize -> 28397912, p50FileSize -> 28397912, numAddedBytes -> 28397912)",null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-10-30T14:01:32.000Z,149468723871717,balaji.raghu@one-consultants.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7c9f753b-0d7e-46ea-8958-e3f9a7f5bf2a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-46b94e65-dfec-408a-8771-9e4a0f8ce2cd"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-fae86fe5-0ad9-4ead-96ad-f1bed54ef126""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(986389444892964),5d3c92fb-2d01-42be-ba1f-599b076cd7ac,1030-095212-b631cnfo-v2n,null,WriteSerializable,true,"Map(numFiles -> 200, numOutputRows -> 1136308, numOutputBytes -> 32802767)",null,Databricks-Runtime/19.13.x-photon-scala2.13


### 8.6 Predictive Optimization
For **Unity Catalog managed tables**, predictive optimization lets Databricks decide when to run maintenance, based on how each table is written and queried:
* `OPTIMIZE` (compaction and liquid clustering, including choosing keys for `CLUSTER BY AUTO`)
* `VACUUM` (with the table's retention settings)
* `ANALYZE` (statistics for data skipping and the optimizer)

It is enabled at account / metastore, catalog or schema level and inherited downwards:
```sql
ALTER CATALOG one_dbx_dev ENABLE PREDICTIVE OPTIMIZATION;      -- or DISABLE / INHERIT
ALTER SCHEMA  one_dbx_dev.default ENABLE PREDICTIVE OPTIMIZATION;
```
Operations it runs are logged in the system table `system.storage.predictive_optimization_operations_history`. Below we only **check** the current status (no change).

In [0]:
%sql
-- Look for the 'Predictive Optimization' row (e.g. ENABLE inherited from METASTORE)
DESCRIBE SCHEMA EXTENDED training.default;

database_description_item,database_description_value
Catalog Name,training
Namespace Name,default
Comment,Default schema (auto-created)
Location,
Owner,balaji.raghu@one-consultants.com
Properties,"((unity.catalog.managed.delta.defaults.defaultClusterByAuto,true), (unity.catalog.managed.delta.defaults.delta.checkpointPolicy,v2), (unity.catalog.managed.delta.defaults.delta.enableRowTracking,true), (unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_northeurope)


### 8.7 Key Delta Table Properties
| Property | Default | Purpose |
|---|---|---|
| `delta.targetFileSize` | auto-tuned | Target file size for OPTIMIZE / writes (e.g. `128mb`) |
| `delta.tuneFileSizesForRewrites` | auto | Smaller files for tables with frequent MERGE/UPDATE |
| `delta.autoOptimize.optimizeWrite` | varies | Optimized writes |
| `delta.autoOptimize.autoCompact` | varies | Auto compaction |
| `delta.enableDeletionVectors` | true (new tables) | Soft-delete rows instead of rewriting files on UPDATE/DELETE/MERGE |
| `delta.enableChangeDataFeed` | false | Record row-level changes |
| `delta.enableRowTracking` | true on new UC managed tables | Stable row IDs for incremental processing |
| `delta.checkpointInterval` | 10 | Commits between checkpoints |
| `delta.logRetentionDuration` | interval 30 days | How long commit history is kept |
| `delta.deletedFileRetentionDuration` | interval 7 days | How long removed data files are kept before VACUUM can delete them |
| `delta.dataSkippingNumIndexedCols` / `delta.dataSkippingStatsColumns` | 32 / unset | Columns with file statistics |
| `delta.parquet.compression.codec` | zstd (new managed tables) | Parquet compression |

In [0]:
%sql
ALTER TABLE tpc_orders_small SET TBLPROPERTIES (
  'delta.targetFileSize'               = '128mb',
  'delta.logRetentionDuration'         = 'interval 30 days',
  'delta.deletedFileRetentionDuration' = 'interval 7 days'
);
SHOW TBLPROPERTIES tpc_orders_small;

key,value
clusterByAuto,true
clusteringColumns,[]
delta.autoOptimize.autoCompact,auto
delta.autoOptimize.optimizeWrite,true
delta.checkpointPolicy,v2
delta.deletedFileRetentionDuration,interval 7 days
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.clustering,supported


## 9. VACUUM
* `VACUUM` permanently deletes data files no longer referenced by the current version and older than the retention period (default 7 days). After VACUUM, time travel to versions older than the retention period is no longer possible.
* `DRY RUN` lists files that would be deleted without removing anything. Because our files are only minutes old, the list is expected to be empty.
* `delta.deletedFileRetentionDuration` (default `interval 7 days`) controls how long removed files are kept; `delta.logRetentionDuration` (default `interval 30 days`) controls how long commit history is kept.
* **FULL** mode (default) lists the whole table directory to find unreferenced files. **LITE** mode uses only the transaction log, so it is much faster on large tables but can miss files written by failed jobs.
* Never set retention below the longest-running query or stream on the table; Databricks blocks `RETAIN` values below 7 days unless `spark.databricks.delta.retentionDurationCheck.enabled` is disabled.

In [0]:
%sql
VACUUM tpc_orders_small DRY RUN;

path


In [0]:
%sql
-- The 200 small files removed by OPTIMIZE are still in storage (for time travel).
-- They become eligible once older than 168 hours (7 days); today this list is empty.
VACUUM tpc_orders_small RETAIN 168 HOURS DRY RUN;

path


In [0]:
%sql
-- LITE mode finds candidates from the transaction log only (no directory listing)
VACUUM tpc_orders LITE DRY RUN;

path


In [0]:
%sql
-- Retention settings that decide what VACUUM and log cleanup may remove
SHOW TBLPROPERTIES tpc_orders_small ('delta.deletedFileRetentionDuration');

key,value
delta.deletedFileRetentionDuration,interval 7 days


In [0]:
%sql
-- Full lifecycle: CTAS -> INSERT -> UPDATE -> DELETE -> MERGE -> OPTIMIZE
DESCRIBE HISTORY tpc_orders_small;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-10-30T14:10:50.000Z,149468723871717,balaji.raghu@one-consultants.com,SET TBLPROPERTIES,"Map(properties -> {""delta.targetFileSize"":""128mb"",""delta.logRetentionDuration"":""interval 30 days"",""delta.deletedFileRetentionDuration"":""interval 7 days""})",null,List(986389444892964),191f3a3e-332c-4cb2-8f43-7d2b44aa07ae,1030-095212-b631cnfo-v2n,3,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-photon-scala2.13
3,2026-10-30T14:04:06.000Z,149468723871717,balaji.raghu@one-consultants.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(986389444892964),ef712025-5096-4c1a-b2ab-43ecfad79118,1030-095212-b631cnfo-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1141556, numOutputBytes -> 28226042)",null,Databricks-Runtime/19.13.x-photon-scala2.13
2,2026-10-30T14:03:56.000Z,149468723871717,balaji.raghu@one-consultants.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""auto""})",null,List(986389444892964),bb7486a9-ba38-460e-83a7-d63837c4ee03,1030-095212-b631cnfo-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-photon-scala2.13
1,2026-10-30T14:02:04.000Z,149468723871717,balaji.raghu@one-consultants.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(986389444892964),7433fc7c-b834-4b97-b361-d41464fc2d58,1030-095212-b631cnfo-v2n,0,SnapshotIsolation,false,"Map(numRemovedFiles -> 200, numRemovedBytes -> 32802767, p25FileSize -> 28397912, numDeletionVectorsRemoved -> 0, minFileSize -> 28397912, numAddedFiles -> 1, maxFileSize -> 28397912, p75FileSize -> 28397912, p50FileSize -> 28397912, numAddedBytes -> 28397912)",null,Databricks-Runtime/19.13.x-photon-scala2.13
0,2026-10-30T14:01:32.000Z,149468723871717,balaji.raghu@one-consultants.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7c9f753b-0d7e-46ea-8958-e3f9a7f5bf2a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-46b94e65-dfec-408a-8771-9e4a0f8ce2cd"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-fae86fe5-0ad9-4ead-96ad-f1bed54ef126""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(986389444892964),5d3c92fb-2d01-42be-ba1f-599b076cd7ac,1030-095212-b631cnfo-v2n,null,WriteSerializable,true,"Map(numFiles -> 200, numOutputRows -> 1136308, numOutputBytes -> 32802767)",null,Databricks-Runtime/19.13.x-photon-scala2.13


## 10. Cleanup
Uncomment and run to remove the training objects.

In [0]:
%sql
DROP TABLE IF EXISTS tpc_orders;
DROP TABLE IF EXISTS tpc_orders_partitioned;
DROP TABLE IF EXISTS tpc_orders_clustered;
 DROP TABLE IF EXISTS tpc_log_demo;
 DROP TABLE IF EXISTS tpc_orders_cdf;
 DROP TABLE IF EXISTS tpc_customer;
 DROP TABLE IF EXISTS tpc_orders_small;
 DROP VIEW  IF EXISTS vw_order_summary;
SELECT 'Cleanup statements are commented out' AS note;

note
Cleanup statements are commented out
